# From Effects to Decisions
## P@S Class 19: The Green school on the limits of A/B

You estimated the treatment effect. Now what do you DO with it?

This notebook walks through three moves from today's lecture:

1. **Kalla-Broockman (2018)**: meta-analysis of 49 political field experiments with DerSimonian-Laird random-effects, showing pooled persuasion effect is ~0 in general elections.
2. **Lewis-Rao (2015)**: power simulation showing why ROI CIs stay huge even with millions of person-weeks of data.
3. **Thompson sampling teaser**: bridge to Class 21 (Multi-Armed Bandits) and HW4.

Readings for today:
- Kalla & Broockman (2018). "Minimal Persuasive Effects of Campaign Contact." APSR 112(1).
- Lewis & Rao (2015). "Unfavorable Economics of Measuring Ad Returns." QJE 130(4).
- Offer-Westort, Coppock & Green (2021). "Adaptive Experimental Design." AJPS 65(4).

In [ ]:
# Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)  # reproducibility
plt.style.use('seaborn-v0_8-whitegrid')

# Consistent color palette
BLUE, RED, GREEN, GRAY, ORANGE = '#2980b9', '#c0392b', '#27ae60', '#7f8c8d', '#e67e22'

---
## Section 1: Kalla and Broockman (2018) meta-analysis

Kalla and Broockman collected 49 field experiments of campaign contact in general elections. Each experiment estimates a persuasion effect on vote share. The meta-analytic question: what is the POOLED effect across all these experiments, and is it distinguishable from zero?

They use a **DerSimonian-Laird random-effects model**. The intuition:
- Each study has its own sampling variance (from its own N and design)
- There may be additional between-study heterogeneity (different campaigns, contact modes, voter populations)
- DL estimates both components and combines them into a pooled estimate

We build synthetic data matching the paper's reported characteristics: ~49 experiments, effect sizes in percentage points of vote share, most near zero.

In [ ]:
# Build synthetic Kalla-Broockman-style meta-analysis data
# Matches paper's stated characteristics:
# - 49 experiments of campaign contact in general elections
# - Effect sizes in percentage points of vote share (mostly between -3 and +3)
# - Pooled effect ~0, heterogeneity Q/df ~ 1.15 (consistent with Kalla Table 1)
# - Individual study SEs range from ~0.4 to ~2.5 percentage points

K = 49  # number of studies

# True underlying effect is zero (Kalla-Broockman's finding)
true_tau = 0.0

# Each study has its own standard error, drawn from a realistic range
# Smaller studies have higher SEs; Kalla's studies vary a lot in N
study_se = np.random.uniform(0.4, 2.5, K)  # percentage points

# Observed effects are the true effect plus sampling noise
# For random-effects, also add a small between-study variance component
between_study_sd = 0.3  # small heterogeneity (matches Q/df ~ 1.15)
study_true_effects = true_tau + np.random.normal(0, between_study_sd, K)
study_estimates = study_true_effects + np.random.normal(0, study_se)

# Build dataframe
df = pd.DataFrame({
    'study_id': range(1, K+1),
    'estimate': study_estimates,
    'se': study_se,
    'ci_lo': study_estimates - 1.96 * study_se,
    'ci_hi': study_estimates + 1.96 * study_se,
})

print(f"Generated {K} synthetic experiments")
print(f"Estimate range: [{df['estimate'].min():.2f}, {df['estimate'].max():.2f}] pp")
print(f"SE range:       [{df['se'].min():.2f}, {df['se'].max():.2f}] pp")
df.head()

In [ ]:
# DerSimonian-Laird random-effects meta-analysis
# Step 1: fixed-effects weights (inverse variance)
w_fe = 1.0 / df['se']**2

# Fixed-effects pooled estimate (used to compute Q)
theta_fe = np.average(df['estimate'], weights=w_fe)

# Step 2: compute Q statistic (test for heterogeneity)
Q = np.sum(w_fe * (df['estimate'] - theta_fe)**2)
k = len(df)
dof = k - 1

# Step 3: estimate between-study variance (tau-squared)
# DL estimator: tau^2 = max(0, (Q - df) / c) where c = sum(w) - sum(w^2)/sum(w)
c = w_fe.sum() - (w_fe**2).sum() / w_fe.sum()
tau_sq = max((Q - dof) / c, 0)

# Step 4: random-effects weights (inverse of within + between variance)
w_re = 1.0 / (df['se']**2 + tau_sq)

# Pooled random-effects estimate
theta_re = np.average(df['estimate'], weights=w_re)
se_re = np.sqrt(1.0 / w_re.sum())
ci_lo_re = theta_re - 1.96 * se_re
ci_hi_re = theta_re + 1.96 * se_re
p_val = 2 * (1 - stats.norm.cdf(abs(theta_re / se_re)))

# Heterogeneity statistics
I_sq = max(0, (Q - dof) / Q) * 100

print(f"=== DerSimonian-Laird Random-Effects Meta-Analysis ===")
print(f"Studies: k = {k}")
print(f"Pooled effect: {theta_re:.3f} pp (SE = {se_re:.3f})")
print(f"95% CI: [{ci_lo_re:.3f}, {ci_hi_re:.3f}] pp")
print(f"p-value: {p_val:.3f}")
print(f"")
print(f"Heterogeneity:")
print(f"  Q(df={dof}) = {Q:.2f}")
print(f"  tau^2 = {tau_sq:.3f}")
print(f"  I^2 = {I_sq:.1f}%")

In [ ]:
# Forest plot of the 49 synthetic experiments
fig, ax = plt.subplots(figsize=(10, 14))

# Sort by effect size for readability
df_sorted = df.sort_values('estimate').reset_index(drop=True)

for i, row in df_sorted.iterrows():
    ax.plot([row['ci_lo'], row['ci_hi']], [i, i],
            color=BLUE, linewidth=1.2, alpha=0.6)  # CI whiskers
    ax.plot(row['estimate'], i, 'o', color=BLUE, markersize=4)  # point estimate

# Meta-analytic diamond below all studies
diamond_y = len(df_sorted) + 2
ax.plot(theta_re, diamond_y, 'D', color='black', markersize=12, zorder=5)
ax.plot([ci_lo_re, ci_hi_re], [diamond_y, diamond_y], 'k-', linewidth=2.5)

# Zero line
ax.axvline(0, color=GRAY, linestyle='--', alpha=0.6)

ax.set_xlabel('Treatment Effect on Vote Share (percentage points)', fontsize=12)
ax.set_title(
    f'49 Political Field Experiments: Meta-Analysis\n'
    f'Pooled effect = {theta_re:.2f} pp  '
    f'(95% CI [{ci_lo_re:.2f}, {ci_hi_re:.2f}], p = {p_val:.2f})',
    fontsize=13
)
ax.set_yticks([])
ax.set_xlim(-7, 7)

ax.annotate(
    'Pooled\n(DerSimonian-Laird)',
    xy=(theta_re, diamond_y),
    xytext=(3.5, diamond_y - 8),
    fontsize=10,
    arrowprops=dict(arrowstyle='->', color='black')
)

plt.tight_layout()
plt.show()

**What you should see:** 49 individual effect estimates (blue dots) scattered around zero. Most 95% CIs cross zero. The meta-analytic diamond (black) sits essentially on zero with a tight CI. Individual campaigns can show small positive or negative results due to noise, but the POOLED effect across all these experiments is indistinguishable from zero.

This is Kalla and Broockman's core finding. The RCTs are not broken: they are correctly measuring a zero underlying effect. General-election persuasion from campaign contact is ~0.

**Where persuasion effects DO appear** (from the paper):
- Early primaries (before voters form opinions)
- Low-information races (school board, state legislature)
- Unusual candidates (partisan ID fails to predict)
- Early contact (weeks before election day)

---
## Section 2: Lewis-Rao (2015) power simulation

Kalla-Broockman measured effects that happen to be zero. Lewis-Rao show a different failure mode: even when effects might be nonzero, the NOISE is so large that we cannot measure them reliably.

Their setup: 25 digital ad experiments across major retailers. Total ad spend: $2.8M. Sample sizes: millions of person-weeks each. Outcome: consumer spending (which is extremely spiky: most people buy nothing, some buy a lot).

They show that the 95% confidence interval on ROI stays wider than 100 percentage points even with these huge samples. A CI of that width cannot distinguish "+50% profit" from "-50% profit." The experiment is statistically helpless.

Below we reproduce the key logic with synthetic data.

In [ ]:
# Simulate per-user consumer outcomes with realistic spiky distribution
# Most users spend $0; a small fraction spends a lot
def generate_users(n, lift_pct=0.0):
    """Generate n users' purchase outcomes. Optional treatment lift (percentage)."""
    # Purchase probability: ~3% of users buy anything
    purchase_prob = 0.03
    # Lift is applied multiplicatively to the probability
    effective_prob = purchase_prob * (1.0 + lift_pct / 100.0)
    buys = np.random.binomial(1, effective_prob, n)
    # Purchase amounts: lognormal with median ~$30, heavy right tail up to $1000+
    log_means = np.random.normal(np.log(30), 1.0, n)
    purchases = np.exp(log_means) * buys  # zero if they did not buy
    return purchases

# Single experiment at a given sample size per arm
def run_experiment(n_per_arm, true_lift_pct=0.5):
    ctrl = generate_users(n_per_arm, lift_pct=0.0)
    treat = generate_users(n_per_arm, lift_pct=true_lift_pct)

    # Estimate lift: difference in means divided by control mean
    lift_hat = (treat.mean() - ctrl.mean()) / ctrl.mean() * 100.0

    # SE of the lift estimate (delta method)
    se_diff = np.sqrt(treat.var()/n_per_arm + ctrl.var()/n_per_arm)
    se_lift = se_diff / ctrl.mean() * 100.0

    return lift_hat, se_lift

# Quick single-experiment sanity check at 100K per arm
lift, se = run_experiment(100_000, true_lift_pct=0.5)
print(f"N = 100K per arm, true lift = 0.5%")
print(f"  Estimated lift: {lift:.2f}% (SE = {se:.2f}%)")
print(f"  95% CI: [{lift - 1.96*se:.2f}%, {lift + 1.96*se:.2f}%]")
print(f"  CI width: {2*1.96*se:.1f} percentage points")

In [ ]:
# Sweep sample sizes from 10K to 10M per arm
# At each N, run a single experiment and record the CI width
sample_sizes = [10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000, 10_000_000]
results = []

for n in sample_sizes:
    lift_hat, se_lift = run_experiment(n, true_lift_pct=0.5)
    ci_width = 2 * 1.96 * se_lift
    results.append({'n_per_arm': n, 'lift_hat': lift_hat,
                    'se': se_lift, 'ci_width': ci_width})

result_df = pd.DataFrame(results)
print("Sample size sweep (true lift = 0.5%):")
print(result_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# Plot CI width vs sample size (log-log)
fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(result_df['n_per_arm'], result_df['ci_width'],
          'o-', color=RED, markersize=10, linewidth=2, label='95% CI width (pp)')

# Reference: true lift of 0.5% (what we are trying to detect)
ax.axhline(0.5, color=GREEN, linestyle='--', linewidth=2,
           label='True lift = 0.5% (target to detect)')

# Reference: 100pp threshold (Lewis-Rao headline)
ax.axhline(100, color=GRAY, linestyle=':', linewidth=1.5,
           label='100 pp (Lewis-Rao ceiling)')

ax.set_xlabel('Sample size per arm', fontsize=12)
ax.set_ylabel('95% CI width (percentage points)', fontsize=12)
ax.set_title(
    'The Unfavorable Economics of Measuring Ad Returns\n'
    'CI width shrinks slowly; you need 10M+ per arm before it rivals the effect',
    fontsize=13
)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

**What you should see:** The 95% CI width drops as you add sample size, but it drops slowly (as $1/\sqrt{N}$). Even at 10M per arm, the CI width is still ~1 percentage point: still wider than the 0.5 pp effect we are trying to detect.

This is why Google's own ad team (Lewis and Rao both worked there) could not measure their own ad ROI reliably even with enormous samples.

Apply Fisher's N ≈ 16s²/d² rule of thumb: with d ≈ 0.001 (tiny standardized effect), you need roughly 16/0.000001 = 16 million per arm. Matches the empirical result above.

**Two failure modes from today:**

| Paper | Failure mode | Claim |
|---|---|---|
| Kalla-Broockman | empirical | we CAN measure; answer is ~0 |
| Lewis-Rao | epistemic | noise too big; we cannot measure |

---
## Section 3: Thompson sampling teaser

Both nulls above assume STATIC A/B: you commit to 50/50 allocation for the entire experiment. Offer-Westort, Coppock, and Green (2021) argue this wastes data.

If you have 10 arms and one is clearly winning by batch 3, why keep sending 10% of traffic to each losing arm? Thompson sampling uses posterior probabilities to allocate dynamically: shift traffic toward arms that look good, stay on losers only enough to rule them out.

Below: simulate 2-arm bandit with Beta-Binomial posteriors, compare static 50/50 vs Thompson sampling.

This is the bridge to Class 21 (Multi-Armed Bandits) and HW4.

In [ ]:
# 2-arm bandit with clear winner
# Arm A: true success rate 0.05 (5% click-through)
# Arm B: true success rate 0.07 (7% click-through, slightly better)
true_rates = [0.05, 0.07]
n_total = 4000

# Static 50/50 allocation
static_pulls = np.array([n_total // 2, n_total // 2])
static_successes = np.random.binomial(static_pulls, true_rates)

# Thompson sampling: at each step, sample from each arm's posterior
# and pull whichever arm has the highest sampled rate
ts_pulls = np.zeros(2, dtype=int)
ts_successes = np.zeros(2, dtype=int)
ts_alloc_history = []  # fraction of traffic sent to arm B over time

for t in range(n_total):
    # Beta-Binomial posteriors (Beta(1,1) prior)
    sampled_rates = [np.random.beta(ts_successes[a] + 1,
                                     ts_pulls[a] - ts_successes[a] + 1)
                     for a in range(2)]
    # Pull the arm with highest sampled rate
    pull = int(np.argmax(sampled_rates))
    ts_pulls[pull] += 1
    ts_successes[pull] += np.random.binomial(1, true_rates[pull])
    # Record allocation to arm B so far
    ts_alloc_history.append(ts_pulls[1] / (t+1))

print("=== Results after", n_total, "pulls ===")
print(f"Static:   Arm A pulled {static_pulls[0]}, Arm B pulled {static_pulls[1]}")
print(f"          Successes: A={static_successes[0]}, B={static_successes[1]}")
print(f"          Total successes: {static_successes.sum()}")
print()
print(f"Thompson: Arm A pulled {ts_pulls[0]}, Arm B pulled {ts_pulls[1]}")
print(f"          Successes: A={ts_successes[0]}, B={ts_successes[1]}")
print(f"          Total successes: {ts_successes.sum()}")
print()
print(f"Regret reduction: Thompson got {ts_successes.sum() - static_successes.sum()} more successes")

In [ ]:
# Plot: allocation to winning arm over time
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(range(n_total), ts_alloc_history,
        color=GREEN, linewidth=2, label='Thompson sampling')
ax.axhline(0.5, color=RED, linestyle='--', linewidth=2, label='Static 50/50')

ax.set_xlabel('Time step (interaction number)', fontsize=12)
ax.set_ylabel('Fraction of traffic to Arm B (true winner)', fontsize=12)
ax.set_title(
    'Thompson Sampling Shifts Traffic to the Winner\n'
    '(Arm A = 5%, Arm B = 7%)',
    fontsize=13
)
ax.set_ylim(0, 1)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

**What you should see:** Thompson sampling starts around 50/50 (no evidence yet), then shifts traffic to Arm B as evidence accumulates that it is the winner. By the end of the simulation, 70-90% of traffic is going to the winner. Static 50/50 is oblivious to the evidence.

Same total budget, more wins. That is the adaptive move that Offer-Westort, Coppock, and Green advocate.

**Next week (Class 20 Targeting):** what if we do not just want the best AVERAGE arm, but the best arm FOR THIS PERSON? Answer: learn a model of $f_a(X) = E[Y \mid A=a, X]$ and pick $\arg\max_a f_a(X)$ per person.

**Class 21 (Multi-Armed Bandits):** formalize this. HW4 launches.

---

## Key takeaways

1. Kalla-Broockman: 49 political field experiments show pooled persuasion effect ~0. The RCT works; the effect IS zero.
2. Lewis-Rao: even with millions of person-weeks, ROI CIs can stay wider than the effects you want to measure. Power is unfavorable.
3. Offer-Westort-Coppock-Green: static A/B wastes data. Adaptive allocation (Thompson sampling) gets more wins with the same budget.
4. These three papers set up the next six classes: targeting, bandits, contextual bandits, recommender systems, LLMs, and governance.